# Practice Notebook: Ingest a Fake Slide Deck into SQLite

This notebook lets you practice a simple version of a pitch-deck ingestion pipeline:

1. Define a fake slide deck as Python data
2. Normalize it into relational tables
3. Insert it into a SQLite database
4. Run a few analysis queries

The schema is intentionally simple so you can modify it later for your startup readiness project.


## 1) Imports and setup

In [ ]:
import sqlite3
from pathlib import Path
from pprint import pprint
import json

DB_PATH = Path("fake_pitch_deck.db")
if DB_PATH.exists():
    DB_PATH.unlink()  # start fresh each run

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

print(f"Using database: {DB_PATH.resolve()}")

## 2) Create a fake slide deck

This simulates what a parser might produce after extracting content from a PDF or PPTX.


In [ ]:
fake_deck = {
    "deck_id": "deck_001",
    "startup_name": "AetherFleet",
    "sector": "Logistics SaaS",
    "source_file": "aetherfleet_seed_deck.pdf",
    "slides": [
        {
            "slide_number": 1,
            "title": "AetherFleet",
            "section": "Title",
            "summary": "AI dispatch and route optimization for regional carriers.",
            "speaker_notes": "Open with pain point and size of opportunity.",
            "bullets": [
                "Reduce empty miles by 18%",
                "Increase dispatcher throughput",
                "Deploy in less than 2 weeks"
            ],
            "claims": [
                {"claim_text": "Reduce empty miles by 18%", "claim_type": "efficiency", "evidence_strength": 3},
                {"claim_text": "Deploy in less than 2 weeks", "claim_type": "implementation", "evidence_strength": 2}
            ]
        },
        {
            "slide_number": 2,
            "title": "Problem",
            "section": "Problem",
            "summary": "Regional carriers still dispatch with spreadsheets, texts, and tribal knowledge.",
            "speaker_notes": "Explain manual workflow pain and margin pressure.",
            "bullets": [
                "Low route visibility",
                "Manual load assignment",
                "High driver idle time"
            ],
            "claims": [
                {"claim_text": "Dispatch teams lose hours to manual assignment", "claim_type": "pain_point", "evidence_strength": 2}
            ]
        },
        {
            "slide_number": 3,
            "title": "Market",
            "section": "Market",
            "summary": "Large underserved mid-market freight segment.",
            "speaker_notes": "Differentiate TAM, SAM, SOM.",
            "bullets": [
                "TAM: $9.2B",
                "SAM: $1.4B",
                "SOM: $120M"
            ],
            "claims": [
                {"claim_text": "TAM is $9.2B", "claim_type": "market_size", "evidence_strength": 1}
            ]
        },
        {
            "slide_number": 4,
            "title": "Traction",
            "section": "Traction",
            "summary": "Early pilots with paying fleets show strong retention.",
            "speaker_notes": "Show revenue and retention trend.",
            "bullets": [
                "12 paying fleets",
                "$28K MRR",
                "109% net revenue retention"
            ],
            "claims": [
                {"claim_text": "12 paying fleets", "claim_type": "traction", "evidence_strength": 4},
                {"claim_text": "$28K MRR", "claim_type": "revenue", "evidence_strength": 4},
                {"claim_text": "109% NRR", "claim_type": "retention", "evidence_strength": 3}
            ]
        },
        {
            "slide_number": 5,
            "title": "Business Model",
            "section": "Business Model",
            "summary": "Subscription pricing based on number of trucks.",
            "speaker_notes": "Emphasize expansion path.",
            "bullets": [
                "$199 per truck per month",
                "Annual contracts",
                "Setup fee for onboarding"
            ],
            "claims": [
                {"claim_text": "$199 per truck per month", "claim_type": "pricing", "evidence_strength": 4}
            ]
        }
    ]
}

pprint(fake_deck)

## 3) Create SQLite schema

We will use four tables:

- `decks`: one row per pitch deck
- `slides`: one row per slide
- `bullets`: one row per bullet point
- `claims`: one row per extracted claim


In [ ]:
schema_sql = '''
CREATE TABLE decks (
    deck_id TEXT PRIMARY KEY,
    startup_name TEXT NOT NULL,
    sector TEXT,
    source_file TEXT
);

CREATE TABLE slides (
    slide_id INTEGER PRIMARY KEY AUTOINCREMENT,
    deck_id TEXT NOT NULL,
    slide_number INTEGER NOT NULL,
    title TEXT,
    section TEXT,
    summary TEXT,
    speaker_notes TEXT,
    FOREIGN KEY (deck_id) REFERENCES decks(deck_id)
);

CREATE TABLE bullets (
    bullet_id INTEGER PRIMARY KEY AUTOINCREMENT,
    slide_id INTEGER NOT NULL,
    bullet_order INTEGER NOT NULL,
    bullet_text TEXT NOT NULL,
    FOREIGN KEY (slide_id) REFERENCES slides(slide_id)
);

CREATE TABLE claims (
    claim_id INTEGER PRIMARY KEY AUTOINCREMENT,
    slide_id INTEGER NOT NULL,
    claim_text TEXT NOT NULL,
    claim_type TEXT,
    evidence_strength INTEGER,
    FOREIGN KEY (slide_id) REFERENCES slides(slide_id)
);
'''
cur.executescript(schema_sql)
conn.commit()

print("Schema created.")

## 4) Insert the fake deck into the database

In [ ]:
def ingest_deck(conn, deck):
    cur = conn.cursor()

    cur.execute(
        '''
        INSERT INTO decks (deck_id, startup_name, sector, source_file)
        VALUES (?, ?, ?, ?)
        ''',
        (
            deck["deck_id"],
            deck["startup_name"],
            deck.get("sector"),
            deck.get("source_file"),
        )
    )

    for slide in deck["slides"]:
        cur.execute(
            '''
            INSERT INTO slides (deck_id, slide_number, title, section, summary, speaker_notes)
            VALUES (?, ?, ?, ?, ?, ?)
            ''',
            (
                deck["deck_id"],
                slide["slide_number"],
                slide.get("title"),
                slide.get("section"),
                slide.get("summary"),
                slide.get("speaker_notes"),
            )
        )
        slide_id = cur.lastrowid

        for i, bullet in enumerate(slide.get("bullets", []), start=1):
            cur.execute(
                '''
                INSERT INTO bullets (slide_id, bullet_order, bullet_text)
                VALUES (?, ?, ?)
                ''',
                (slide_id, i, bullet)
            )

        for claim in slide.get("claims", []):
            cur.execute(
                '''
                INSERT INTO claims (slide_id, claim_text, claim_type, evidence_strength)
                VALUES (?, ?, ?, ?)
                ''',
                (
                    slide_id,
                    claim.get("claim_text"),
                    claim.get("claim_type"),
                    claim.get("evidence_strength"),
                )
            )

    conn.commit()

ingest_deck(conn, fake_deck)
print("Fake deck ingested.")

## 5) Inspect the tables

In [ ]:
for table in ["decks", "slides", "bullets", "claims"]:
    count = cur.execute(f"SELECT COUNT(*) AS n FROM {table}").fetchone()["n"]
    print(f"{table}: {count} rows")

In [ ]:
rows = cur.execute("SELECT * FROM slides ORDER BY slide_number").fetchall()
for row in rows:
    print(dict(row))

## 6) Example queries

These are the kinds of queries that become useful once many decks are stored.


In [ ]:
query = '''
SELECT
    s.slide_number,
    s.title,
    c.claim_text,
    c.claim_type,
    c.evidence_strength
FROM claims c
JOIN slides s ON c.slide_id = s.slide_id
WHERE c.evidence_strength >= 3
ORDER BY s.slide_number, c.evidence_strength DESC
'''
rows = cur.execute(query).fetchall()
for row in rows:
    print(dict(row))

In [ ]:
query = '''
SELECT
    s.section,
    COUNT(c.claim_id) AS num_claims,
    ROUND(AVG(c.evidence_strength), 2) AS avg_evidence_strength
FROM slides s
LEFT JOIN claims c ON s.slide_id = c.slide_id
GROUP BY s.section
ORDER BY num_claims DESC
'''
rows = cur.execute(query).fetchall()
for row in rows:
    print(dict(row))

## 7) Export the database contents as JSON (optional)

This is useful if you want to inspect what ended up in the relational structure.


In [ ]:
export = {
    "decks": [dict(r) for r in cur.execute("SELECT * FROM decks").fetchall()],
    "slides": [dict(r) for r in cur.execute("SELECT * FROM slides ORDER BY slide_number").fetchall()],
    "bullets": [dict(r) for r in cur.execute("SELECT * FROM bullets ORDER BY slide_id, bullet_order").fetchall()],
    "claims": [dict(r) for r in cur.execute("SELECT * FROM claims ORDER BY slide_id").fetchall()],
}

with open("deck_export.json", "w") as f:
    json.dump(export, f, indent=2)

print("Wrote deck_export.json")

## 8) Practice extensions

Try any of these next:

1. Add a `scores` table for rubric labels
2. Add a `red_flags` table
3. Store per-slide embeddings metadata fields
4. Ingest multiple fake decks and compare sectors
5. Add a `raw_text` column for whole-slide text extraction
6. Build a `core10_section` normalization table


In [ ]:
conn.close()
print("Connection closed.")